# Data Creation

## 1. Imports and Setup

Add the project root to `sys.path` so that `src/` and `config.py` are importable. Output directories are created here to avoid repeated checks in downstream cells.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))

import config
from src.features import load_data, merge_data, drop_null_cols, convert_datetime
from src.utils import full_value_counts

Path(config.PLOTS_PATH).mkdir(parents=True, exist_ok=True)
Path(config.RESULTS_PATH).mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 2. Load Data

Load the raw IEEE-CIS transaction and identity CSVs.

In [ ]:
df_transaction, df_identity = load_data(config.TRANSACTION_PATH, config.IDENTITY_PATH)

print('transaction:', df_transaction.shape)
print('identity:   ', df_identity.shape)

In [ ]:
df_transaction.head()

In [ ]:
df_identity.head()

## 3. Merge

Left join identity onto transactions on `TransactionID`. The identity table covers a subset of transactions; the left join preserves all transactions and leaves identity columns null where no match exists.

In [ ]:
df = merge_data(df_transaction, df_identity)

In [ ]:
n_total = len(df)
n_with_id = df['DeviceType'].notna().sum()
n_without_id = df['DeviceType'].isna().sum()
pct_missing = n_without_id / n_total * 100

n_fraud_no_id = df[df['DeviceType'].isna()][config.TARGET].sum()
fraud_rate_with_id = df[df['DeviceType'].notna()][config.TARGET].mean() * 100
fraud_rate_no_id = df[df['DeviceType'].isna()][config.TARGET].mean() * 100

print(f'total rows:              {n_total:,}')
print(f'rows with identity:      {n_with_id:,}')
print(f'rows without identity:   {n_without_id:,}')
print(f'pct missing identity:    {pct_missing:.1f}%')
print(f'fraud in no-id rows:     {n_fraud_no_id:,}')
print(f'fraud rate (with id):    {fraud_rate_with_id:.2f}%')
print(f'fraud rate (without id): {fraud_rate_no_id:.2f}%')

In [ ]:
labels = ['with identity', 'without identity']
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(labels, [n_with_id, n_without_id], color=['steelblue', 'salmon'])
axes[0].set_title('row counts')
axes[0].set_ylabel('n rows')

axes[1].bar(labels, [fraud_rate_with_id, fraud_rate_no_id], color=['steelblue', 'salmon'])
axes[1].set_title('fraud rate (%)')
axes[1].set_ylabel('fraud rate')

fig.tight_layout()
fig.savefig(config.PLOTS_PATH / 'identity_missingness.png')
plt.show()
plt.close()

Rows with missing identity data are retained. Missingness is treated as signal and handled natively by gradient boosting models during training.

## 4. Null Analysis

Inspect null rates across all columns. Columns above `NULL_THRESHOLD` (80%) are dropped — they carry too little signal to be useful and would complicate imputation.

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False)
top30 = null_pct.head(30)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top30.index[::-1], top30.values[::-1])
ax.set_xlabel('null fraction')
ax.set_title('top 30 columns by null rate')
fig.tight_layout()
fig.savefig(config.PLOTS_PATH / 'null_analysis.png')
plt.show()
plt.close()

In [ ]:
df = drop_null_cols(df, threshold=config.NULL_THRESHOLD)

## 5. DateTime Conversion

`TransactionDT` is seconds elapsed from a reference date (~2017-11-30). Convert to datetime and extract `hour_of_day` and `day_of_week` as time features.

In [ ]:
df = convert_datetime(df)
df[['TransactionDT', 'hour_of_day', 'day_of_week']].head(10)

## 6. Target Analysis

The dataset is heavily imbalanced. Confirm the fraud rate and verify no rows were dropped during cleaning.

In [ ]:
vc = full_value_counts(df, config.TARGET)
print(vc)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(vc.index.astype(str), vc['cnt'])
ax.set_title(f'{config.TARGET} distribution')
ax.set_xlabel(config.TARGET)
ax.set_ylabel('count')
fig.tight_layout()
fig.savefig(config.PLOTS_PATH / 'target_distribution.png')
plt.show()
plt.close()

fraud_rate = df[config.TARGET].mean() * 100
print(f'fraud rate: {fraud_rate:.2f}%')
print('rows dropped: 0 — no rows have been removed at this stage')

## 7. Save

Write the cleaned dataframe to parquet for use in downstream notebooks.

In [ ]:
out = Path(config.RESULTS_PATH) / 'data_clean.parquet'
df.to_parquet(out, index=False)
print(f'saved to {out}')
print(f'final shape: {df.shape}')